In [24]:
import os
import sys
import subprocess
import shutil
import time
import difflib

# ================= 全局配置 =================
MY_COMPILER   = "./build_middle/compiler_middle_end" #使用Release进行测试，目的仅仅是为了让可执行文件运行的快一点
CLANG_PATH    = "clang-18"                           # Clang
RISCV_GCC     = "riscv64-linux-gnu-gcc"              # GCC 链接器
QEMU_PATH     = "qemu-riscv64"                       # Qemu模拟器
SYLIB_SRC     = "./runtime/sylib.c"                  # 运行时库
TEST_DIR      = "./testcases"                        # 测试目录
TEMP_DIR      = "./temp_worker"                      # 临时目录


if __name__ == "__main__":
    # ---------------------------------------------------------
    # 1. 环境初始化
    # ---------------------------------------------------------
    print(">>> Initializing Workspace...")
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR)

    stats = {"AC": 0, "WA": 0, "CE": 0, "AE": 0, "LE": 0, "TLE": 0, "Bad": 0}
    failed_cases = []

    # ---------------------------------------------------------
    # 2. 编译运行时库 (sylib.o)
    # ---------------------------------------------------------
    print(f">>> Compiling Runtime Library ({SYLIB_SRC})...")
    try:
        # 必须加上 -march=rv64gc
        subprocess.run(
            [RISCV_GCC, SYLIB_SRC, "-c", "-o", os.path.join(TEMP_DIR, "sylib.o"), "-O2","-march=rv64gc"],
            check=True, timeout=10, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
        )
    except subprocess.CalledProcessError:
        print("\033[31m[FATAL] sylib compilation failed!\033[0m")
        sys.exit(1)

    # ---------------------------------------------------------
    # 3. 扫描测试用例
    # ---------------------------------------------------------
    test_files = []
    for root, dirs, files in os.walk(TEST_DIR):
        for f in files:
            if f.endswith(".sy"):
                test_files.append(os.path.join(root, f))
    test_files.sort()
    print(f">>> Found {len(test_files)} test cases. Starting test...\n")

    # ---------------------------------------------------------
    # 4. 核心大循环
    # ---------------------------------------------------------
    for file_path in test_files:
        filename = os.path.basename(file_path)

        # 路径准备
        ll_file  = os.path.join(TEMP_DIR, "test.ll")
        asm_file = os.path.join(TEMP_DIR, "test.s")
        exe_file = os.path.join(TEMP_DIR, "test")
        in_file  = file_path[:-3] + ".in"
        out_file = file_path[:-3] + ".out"

        temp_src = os.path.join(TEMP_DIR, "test.c")
        shutil.copy(file_path, temp_src)

        # =======================================================
        # Phase I: 前端编译 (Compiler -> .ll)
        # =======================================================
        cmd_compile = [MY_COMPILER, "-S", "-o", ll_file, temp_src]

        try:
            ret = subprocess.run(cmd_compile, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=60)
            if ret.returncode != 0:
                print(f"\033[31m[CE]\033[0m  {filename}")
                stats["CE"] += 1
                failed_cases.append(filename)
                continue
        except subprocess.TimeoutExpired:
            print(f"\033[33m[TLE]\033[0m {filename} (Frontend Timeout)")
            stats["TLE"] += 1
            failed_cases.append(filename)
            continue

        # =======================================================
        # Phase II: 后端编译 (Clang -> .s)
        # =======================================================
        cmd_clang = [
            CLANG_PATH,
            "-S",
            "--target=riscv64-linux-gnu",
            "-march=rv64gc",
            #"-mabi=lp64d",
            "-O0",
            "-fno-addrsig", #下一步是用gcc链接所以这一步要禁用clang编译生成的相关东西
            ll_file,
            "-o", asm_file
        ]

        try:
            ret = subprocess.run(cmd_clang, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=30)
            if ret.returncode != 0:
                print(f"\033[31m[AE]\033[0m  {filename}")
                stats["AE"] += 1
                failed_cases.append(filename)
                continue
        except subprocess.TimeoutExpired:
            print(f"\033[31m[AE]\033[0m  {filename} (Clang Timeout)")
            stats["AE"] += 1
            continue

        # =======================================================
        # Phase III: 汇编+链接 (GCC -> exe)
        # =======================================================
        cmd_link = [
            RISCV_GCC,
            "-march=rv64gc",
            "-static",  #QEMU必须加这个

            # 输入文件 (汇编 + 库)
            asm_file,
            os.path.join(TEMP_DIR, "sylib.o"),

            "-o", exe_file,

            # 链接库 (SysY 需要)
            "-lm"
        ]

        try:
            ret = subprocess.run(cmd_link, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10)
            if ret.returncode != 0:
                print(f"\033[31m[LE]\033[0m  {filename}")
                # 【↓↓↓ 请务必加上这下面两行 ↓↓↓】
                print("="*20 + " Linker Error Details " + "="*20)
                print(ret.stderr.decode(errors='replace')) # 打印具体报错！！！
                stats["LE"] += 1
                failed_cases.append(filename)
                continue
        except subprocess.TimeoutExpired:
            print(f"\033[31m[LE]\033[0m  {filename} (Link Timeout)")
            stats["LE"] += 1
            continue

        # =======================================================
        # Phase IV: 运行 (QEMU)
        # =======================================================
        cmd_run = [QEMU_PATH, exe_file]
        # 读取 .in 文件作为标准输入
        input_data = b""
        if os.path.exists(in_file):
            with open(in_file, "rb") as f: input_data = f.read()

        start_time = time.time()
        try:
            proc = subprocess.run(
                cmd_run,
                input=input_data,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=120         # 运行时限 120秒，后续不仅仅测试前端就可以缩减这里的时间
            )
            run_output = proc.stdout.decode(errors='replace')
            run_retcode = proc.returncode
        except subprocess.TimeoutExpired:
            print(f"\033[33m[TLE]\033[0m {filename} (Runtime Timeout)")
            stats["TLE"] += 1
            failed_cases.append(filename)
            continue

        elapsed = time.time() - start_time

        # =======================================================
        # Phase V: 结果比对 (Strict Difflib Mode)
        # =======================================================
        if not os.path.exists(out_file):
            print(f"\033[35m[Bad]\033[0m {filename} (No .out file)")
            stats["Bad"] += 1
            continue

        # 1. 构造实际输出 (dump_str)
        # 这里必须手动模拟 SysY 的要求：把返回值拼接在最后
        dump_str = run_output
        if dump_str and not dump_str.endswith('\n'):
            dump_str += "\n"
        dump_str += str(run_retcode) + "\n"

        # 2. 读取预期输出 (std_output)
        with open(out_file, "r", errors='ignore') as f:
            std_output = f.read()

        # 3. 使用 Difflib 严格对比
        diff = difflib.unified_diff(dump_str.splitlines(), std_output.splitlines(), lineterm='')

        # 只要 diff 生成器不为空，说明有差异
        if len(list(diff)) != 0:
            print(f"\033[31m[WA]\033[0m {filename}")
            stats["WA"] += 1
            failed_cases.append(filename)
        else:
            print(f"\033[32m[AC]\033[0m {filename} ({elapsed:.3f}s)")
            stats["AC"] += 1

    # ---------------------------------------------------------
    # 5. 最终统计
    # ---------------------------------------------------------
    print("\n" + "="*40)
    print("TEST SUMMARY")
    print("="*40)
    print(f"Total Cases: {len(test_files)}")
    print(f"\033[32mAC (Accepted):       {stats['AC']}\033[0m")
    print(f"\033[31mWA (Wrong Answer):   {stats['WA']}\033[0m")
    print(f"\033[31mCE (Compile Error):  {stats['CE']}\033[0m")
    print(f"\033[31mAE (Assembler Error):{stats['AE']}\033[0m")
    print(f"\033[31mLE (Link Error):     {stats['LE']}\033[0m")
    print(f"\033[33mTLE (Timeout):       {stats['TLE']}\033[0m")
    print("="*40)


>>> Initializing Workspace...
>>> Compiling Runtime Library (./runtime/sylib.c)...
>>> Found 382 test cases. Starting test...

[AC] 2024-0ZE-13.sy (2.214s)
[AC] 2024-1WO-11.sy (21.038s)
[AC] 2024-2KM-21.sy (2.098s)
[AC] 2024-7UJ-12.sy (0.319s)
[AC] 2024-8NL-18.sy (2.516s)
[AC] 2024-931-6.sy (0.972s)
[AC] 2024-BAV-3.sy (96.968s)
[AC] 2024-C64-14.sy (25.548s)
[AC] 2024-CH6-24.sy (1.238s)
[AC] 2024-DCV-7.sy (3.513s)
[AC] 2024-FAF-19.sy (9.887s)
[AC] 2024-FVJ-15.sy (3.760s)
[AC] 2024-H17-1.sy (17.905s)
[AC] 2024-I4W-2.sy (3.236s)
[AC] 2024-JVU-10.sy (21.042s)
[AC] 2024-K0I-17.sy (5.499s)
[AC] 2024-KVV-20.sy (54.552s)
[AC] 2024-LN4-9.sy (8.584s)
[AC] 2024-MXP-16.sy (5.080s)
[AC] 2024-NCY-23.sy (9.671s)
[AC] 2024-NKK-22.sy (12.071s)
[AC] 2024-NT1-4.sy (9.960s)
[AC] 2024-PKP-8.sy (32.462s)
[AC] 2024-U03-25.sy (22.500s)
[AC] 2024-VTF-5.sy (2.361s)
[AC] 2024-1RK-26.sy (9.893s)
[AC] 2024-2D0-22.sy (2.265s)
[AC] 2024-51P-30.sy (38.890s)
[AC] 2024-5HA-8.sy (10.517s)
[AC] 2024-5NG-2.sy (1.680s)
[AC